# AI 기본법 RAG 실습 — 원문에서 Grounded Answer까지 한 번에

### 이번 실습의 위치

```text
01 원문 열어보기 (HWPX)
    ↓
02 법률 구조 파싱 (장·절·조·항·호)
    ↓
03 Chunking: 500자 고정 vs 조문 단위
    ↓
04 임베딩 + Qdrant 저장
    ↓
05 Dense 검색                ← 500자 분할(S0), 조문 단위(S1)
    ↓
06 BM25 + RRF (Hybrid)    ← Hybrid(S2)
    ↓
07 Query Rewriting + Multi-Query     ← Query Rewriting(S3)
    ↓
08 Reranking                     ← Reranking(S4)
    ↓
09 Context: 정의·참조·역참조  ← 정의·참조(C1), 역참조(C2)
    ↓
10 Grounded Answer + 출처
    ↓
11 Golden Test Set 40문항으로 전체 비교
```

이 노트북 하나로 **각 단계에서 무엇이 문제였고, 무엇을 바꿨고, 결과가 어떻게 달라졌는지**를 같은 질문으로 직접 비교한다.

이번 프로젝트가 답해야 하는 세 질문:

> 1. 이 답변은 어떤 문서를 근거로 만들어졌는가?
> 2. 필요한 법률 조항을 제대로 검색했는가?
> 3. 검색 품질을 어떻게 개선할 수 있는가?

# 0. 환경 설정

- Qdrant가 떠 있어야 한다: `cd docker-qdrant && docker compose up -d`
- Reranker 모델 `Qwen/Qwen3-Reranker-0.6B`는 로컬 GPU에서 돈다 (크레딧 0)

### MonoRouter 크레딧 아끼기

LLM 호출(Query Rewriting·답변)은 **월 크레딧**을 쓴다. 그래서 기본값은 `LIVE = False`:

| 단계 | LIVE=False (기본) | LIVE=True |
|---|---|---|
| 파싱·Chunking·BM25·RRF·Reranking·Context | 직접 실행 (크레딧 0) | 직접 실행 |
| 임베딩 (질문 벡터) | 처음 한 번만 호출 후 `notebook/.cache`에 저장 | 매번 호출 |
| Query Rewriting·답변 | 평가 때 저장한 결과(`eval/details.json`)를 읽음 | 실제 호출 (질문당 약 90 크레딧) |

In [1]:
import json, os, re, sys, zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

# 노트북은 notebook/ 에서 실행되므로 프로젝트 루트로 이동한다.
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

LIVE = False  # True로 바꾸면 LLM을 실제로 호출한다 (크레딧 소모)

from common.credits import credits
c = credits()
print(f"남은 MonoRouter 크레딧: {c['remaining_credits']:,} / {c['max_credits']:,}")

남은 MonoRouter 크레딧: 9,616 / 30,000


In [2]:
# 실습에 쓸 질문 4개 — Golden Test Set(docs/eval_set/golden_set.jsonl)에서 단계별 차이가 잘 보이는 것
GOLDEN = {json.loads(l)["id"]: json.loads(l) for l in open("docs/eval_set/golden_set.jsonl")}
DETAILS = json.load(open("eval/details.json"))  # 평가 실행 때 저장한 Query Rewriting·검색 결과

DEMO = ["g02", "g04", "g16", "g12"]
for gid in DEMO:
    g = GOLDEN[gid]
    print(f"{gid} [{g['type']}] {g['question']}")
    print(f"     정답 조문: {', '.join('제' + n + '조' for n in g['gold_articles'])}")

g02 [일상어] 우리 가게 상담 챗봇이 AI라는 걸 손님한테 미리 말해줘야 해? 안 하면 어떻게 돼?
     정답 조문: 제31조, 제43조
g04 [일상어] 은행 AI가 내 대출을 거절했는데, 왜 그런지 설명해 달라고 할 수 있어?
     정답 조문: 제3조, 제34조, 제2조
g16 [정의/세부] 고영향 인공지능에 해당하는 영역에는 어떤 것들이 있어?
     정답 조문: 제2조
g12 [multi-hop] 해외 AI 사업자가 국내대리인을 안 두면 어떻게 되고, 국내대리인은 구체적으로 어떤 일을 대신해?
     정답 조문: 제36조, 제43조, 제32조, 제33조, 제34조


# 1. 원문 열어보기 — HWPX는 사실 zip 파일

RFP는 Docling으로 문서를 읽으라고 했지만, 우리 원본은 **HWPX(한글 문서)** 다.

```text
ai_basic_law_21311_20260721.hwpx  (= zip)
 ├── Contents/section0.xml   ← 본문 (문단마다 <hp:p>, 글자는 <hp:t>)
 ├── Contents/header.xml
 └── ...
```

HWPX는 이미 **문단 단위 XML**이라 조·항·호가 줄마다 나뉘어 있다.
PDF로 바꿔 다시 추출할 이유가 없다 → 표준 라이브러리 `zipfile` + `xml.etree`로 충분하다.

In [3]:
HWPX = Path("data/ai_basic_law/ai_basic_law_21311_20260721.hwpx")
z = zipfile.ZipFile(HWPX)
print([n for n in z.namelist() if n.startswith("Contents/")])

xml_text = z.read("Contents/section0.xml").decode("utf-8")
i = xml_text.find("제1조")
print("\n--- XML 원문 일부 ---")
print(xml_text[i - 300:i + 120])

['Contents/content.hpf', 'Contents/header.xml', 'Contents/section0.xml']

--- XML 원문 일부 ---
 [법률 제21311호, 2026. 1. 20., 일부개정]</hp:t><hp:t><hp:lineBreak/></hp:t></hp:run></hp:p><hp:p id="0" paraPrIDRef="18" styleIDRef="0" pageBreak="0" columnBreak="0" merged="0"><hp:run charPrIDRef="5"><hp:t><hp:lineBreak/>      제1장 총칙</hp:t></hp:run><hp:run charPrIDRef="5"><hp:t><hp:lineBreak/>            제1조(목적)</hp:t></hp:run><hp:run charPrIDRef="5"><hp:t><hp:lineBreak/>            제2조(정의)</hp:t></hp:run><hp:run charPrIDR


In [4]:
# 문단(<hp:p>) 하나 = 한 줄. 그 안의 <hp:t> 글자를 이어 붙이면 줄 텍스트가 된다.
NS = "{http://www.hancom.co.kr/hwpml/2011/paragraph}"
root = ET.fromstring(z.read("Contents/section0.xml"))
lines = ["".join(t.text or "" for t in p.iter(NS + "t")) for p in root.iter(NS + "p")]

print(f"문단 {len(lines)}개\n")
for n, line in enumerate(lines[:26]):
    print(f"{n:3d} | {line[:90]}")

문단 409개

  0 | 
  1 | 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법( 약칭: 인공지능기본법)
  2 | [시행 2026. 7. 21.] [법률 제21311호, 2026. 1. 20., 일부개정]
  3 | 
  4 | 
  5 | 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법( 약칭: 인공지능기본법)
  6 | 인공지능 발전과 신뢰 기반 조성 등에 관한 기... 법제처-  /  - 국가법령정보센터[시행 2026. 7. 21.] [법률 제21311호, 2026. 1. 20
  7 | 인공지능 발전과 신뢰 기반 조성 등에 관한 기...
  8 | 인공지능 발전과 신뢰 기반 조성 등에 관한 기...
  9 | 
 10 | 
 11 |  법제처-  /  - 국가법령정보센터
 12 |  법제처
 13 | -  /  -
 14 |  국가법령정보센터
 15 | 
 16 | 과학기술정보통신부(인공지능안전신뢰정책과) 044-202-6293
 17 | 
 18 |        제1장 총칙
 19 | 제1조(목적) 이 법은 인공지능의 건전한 발전과 신뢰 기반 조성에 필요한 기본적인 사항을 규정함으로써 국민의 권익과 존엄성을 보호하고 국민의 삶의 질 향상과 국가
 20 | 제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다.<개정 2026. 1. 20.>
 21 |   1. “인공지능”이란 학습, 추론, 지각, 판단, 언어의 이해 등 인간이 가진 지적 능력을 전자적 방법으로 구현한 것을 말한다.
 22 |   2. “인공지능시스템”이란 다양한 수준의 자율성과 적응성을 가지고 주어진 목표를 위하여 실제 및 가상환경에 영향을 미치는 예측, 추천, 결정 등의 결과물을 추
 23 |   3. “인공지능기술”이란 인공지능을 구현하기 위하여 필요한 하드웨어ㆍ소프트웨어 기술 또는 그 활용 기술을 말한다.
 24 |   4. “고영향 인공지능”이란 사람의 생명, 신체의 안전 및 기본권에 중대한 영향을 미치거나 위험을 초래할 우려가 있는 인공지능시스템으로서 다음 각

### 무엇이 보이나

- 0~17줄: 법령명 반복, `법제처 - / - 국가법령정보센터`, 담당부서 → **본문이 아니다**
- `       제1장 총칙` → 장 제목. 본문이 아니라 **위치 정보(breadcrumb)** 다
- `제1조(목적) …` → 조문 시작
- `  1. "인공지능"이란 …` → 들여쓰기 2칸 + 숫자 = **호**
- `    가. …` → 들여쓰기 4칸 + 한글 = **목**
- `<개정 2026. 1. 20.>` → 개정 표시. 법 내용이 아니므로 지운다

# 2. 법률 구조 파싱

줄 머리 모양만으로 구조를 판별한다.

| 줄 머리 | 의미 | 정규식 |
|---|---|---|
| `제31조(인공지능 투명성 확보 의무)` | 조 | `^제(\d+)조(?:의(\d+))?\(` |
| `①`, `②` … | 항 | 동그라미 숫자 |
| `  1.`, `  4의2.` | 호 | `^\d+(의\d+)?\.` + 들여쓰기 2칸 |
| `    가.` | 목 | `^[가-힣]\.` + 들여쓰기 4칸 |
| `제N장`, `제N절` | 위치 정보 | 본문에서 분리 |
| `부칙 <법률 …>` | **여기서 멈춤** | 부칙에도 제1조~가 있어 번호가 겹친다 |

In [5]:
# 직접 짜 보는 최소 파서 — 조문 시작 줄만 찾아 본다
ARTICLE_RE = re.compile(r"^제(\d+)조(?:의(\d+))?\(([^)]+)\)")

found = []
for line in lines:
    s = line.strip()
    if s.startswith("부칙"):
        break                      # 부칙 이후는 본문이 아니다
    m = ARTICLE_RE.match(s)
    if m:
        found.append(f"제{m.group(1)}조" + (f"의{m.group(2)}" if m.group(2) else "") + f"({m.group(3)})")

print(len(found), "개 조문")
print(found[:5], "...", found[-3:])

46 개 조문
['제1조(목적)', '제2조(정의)', '제3조(기본원칙 및 국가 등의 책무)', '제4조(적용범위)', '제5조(다른 법률과의 관계)'] ... ['제41조(벌칙 적용에서 공무원 의제)', '제42조(벌칙)', '제43조(과태료)']


### 시행착오 — 실제로 걸렸던 것

1. `제N장` 줄을 지우지 않으면 **앞 조문 끝에 "제2장 …"이 붙는다**
2. `부칙`에서 멈추지 않으면 부칙의 제1조가 본문 제1조와 섞인다
3. 개정 표시 정규식이 `<개정 …>`만 지워서 제7조의 **`[제목개정 2026. 1. 20.]`이 남았다**
4. 가지조문을 `제17의2조`로 만들었다 → 법령 표기는 **`제17조의2`**

3·4번은 아래의 **법제처 Open API 대조**가 잡아냈다. 이 모든 처리를 묶은 것이 `rag/loader.py`다.

In [6]:
from rag.loader import load, units

ARTS = load()
BY = {a["no"]: a for a in ARTS}
print(len(ARTS), "개 조문,", sum(len(a["text"]) for a in ARTS), "자")

a = BY["43"]
print("\n위치:", a["crumb"])
print(a["text"])
print("\n인용 가능한 단위:", [u["label"] for u in units(a)])

46 개 조문, 25272 자

위치: 제6장 벌칙
제43조(과태료) ① 다음 각 호의 어느 하나에 해당하는 자에게는 3천만원 이하의 과태료를 부과한다.
  1. 제31조제1항을 위반하여 고지를 이행하지 아니한 자
  2. 제36조제1항을 위반하여 국내대리인을 지정하지 아니한 자
  3. 제40조제3항에 따른 중지명령이나 시정명령을 이행하지 아니한 자
  ② 제1항에 따른 과태료는 대통령령으로 정하는 바에 따라 과학기술정보통신부장관이 부과ㆍ징수한다.

인용 가능한 단위: ['제43조①', '제43조②', '제43조①1', '제43조①2', '제43조①3']


In [7]:
# 검증: 법제처 Open API로 받은 46개 조문과 글자 단위로 같은가?
api = {x["article"]: x for x in json.load(open("data/ai_basic_law/articles.json"))}

def norm(s):  # 공백·따옴표·개정표시는 표기 차이라 지우고 비교
    s = re.sub(r"<(?:개정|신설)[^>]*>|\[[^\]]*(?:개정|신설|시행일)[^\]]*\]", "", s)
    return re.sub(r"[\s\"'“”‘’]", "", s)

diff = [x["article"] for x in ARTS if norm(x["text"]) != norm(api[x["article"]]["content"])]
print("Open API와 다른 조문:", diff or "없음 — 46개 전부 일치")

Open API와 다른 조문: 없음 — 46개 전부 일치


# 3. Chunking — 500자 고정 vs 조문 단위

```text
[500자 고정 분할]                       [조문 단위]
…제42조(벌칙) … 벌금에 처한다.          ┌ 제42조(벌칙) 전체
제43조(과태료) ① 다음 각 호의 …  ✂      ├ 제43조(과태료) 전체
──────────── 500자 경계 ────────────    │   ① …3천만원 이하의 과태료
  1. 제31조제1항을 위반하여 …            │   1. 제31조제1항 위반 …
  2. 제36조제1항을 위반하여 …            │   2. 제36조제1항 위반 …
```

500자 창은 **"3천만원 과태료"와 "무엇을 어겼을 때"를 다른 청크로 갈라 놓는다.**
법의 최소 의미 단위는 조문이므로 조문 1개 = 청크 1개로 자른다.
예외: 제2조(정의 12개, 약 1,900자)는 정의마다 주제가 달라 **호 단위**로 쪼개고, 앞에 "이 법에서 사용하는 용어의 뜻은 다음과 같다."를 붙인다.

In [8]:
from rag.chunker import chunks, naive_chunks

STRUCT = chunks(ARTS)        # 조문 단위 (제2조만 호 단위)
NAIVE = naive_chunks(ARTS)   # 500자 고정, 50자 겹침

print(f"조문 단위: {len(STRUCT)}개 청크 (제2조 호 단위 {sum(c['no'] == '2' for c in STRUCT)}개)")
print(f"500자 고정: {len(NAIVE)}개 청크, 그중 조문 경계를 가로지르는 청크 {sum(len(c['nos']) > 1 for c in NAIVE)}개")

조문 단위: 57개 청크 (제2조 호 단위 12개)
500자 고정: 57개 청크, 그중 조문 경계를 가로지르는 청크 39개


In [9]:
# 제43조(과태료)가 500자 고정 분할에서 어떻게 잘리는지
for c in NAIVE:
    if "43" in c["nos"]:
        print(f"--- 걸친 조문 {c['nos']} ---")
        print(c["text"][-260:] if c["nos"][0] != "43" else c["text"][:260])
        print()

--- 걸친 조문 ['41', '42', '43'] ---
는 3년 이하의 징역 또는 3천만원 이하의 벌금에 처한다.

제43조(과태료) ① 다음 각 호의 어느 하나에 해당하는 자에게는 3천만원 이하의 과태료를 부과한다.
  1. 제31조제1항을 위반하여 고지를 이행하지 아니한 자
  2. 제36조제1항을 위반하여 국내대리인을 지정하지 아니한 자
  3. 제40조제3항에 따른 중지명령이나 시정명령을 이행하지 아니한 자
  ② 제1항에 따른 과태료는 대통령령으로 정하는 바에 따라 과학기술정보통신부장관이 부과ㆍ징수한다.





In [10]:
# 조문 단위 청크는 이렇게 생겼다 — 앞의 [장 > 절 > 조(제목)]가 위치 정보
print(next(c for c in STRUCT if c["no"] == "43")["text"])
print()
print(next(c for c in STRUCT if c["label"] == "제2조4")["text"][:200], "…")

[제6장 벌칙 > 제43조(과태료)]
제43조(과태료) ① 다음 각 호의 어느 하나에 해당하는 자에게는 3천만원 이하의 과태료를 부과한다.
  1. 제31조제1항을 위반하여 고지를 이행하지 아니한 자
  2. 제36조제1항을 위반하여 국내대리인을 지정하지 아니한 자
  3. 제40조제3항에 따른 중지명령이나 시정명령을 이행하지 아니한 자
  ② 제1항에 따른 과태료는 대통령령으로 정하는 바에 따라 과학기술정보통신부장관이 부과ㆍ징수한다.

[제1장 총칙 > 제2조(정의)] 이 법에서 사용하는 용어의 뜻은 다음과 같다.
4. “고영향 인공지능”이란 사람의 생명, 신체의 안전 및 기본권에 중대한 영향을 미치거나 위험을 초래할 우려가 있는 인공지능시스템으로서 다음 각 목의 어느 하나의 영역에서 활용되는 것을 말한다.
  가. 「에너지법」 제2조제1호에 따른 에너지의 공급
  나. 「먹는물관리법」  …


# 4. 임베딩 + Qdrant 저장

```text
청크 텍스트 ──(text-embedding-3-small)──▶ 1536차원 벡터 ──▶ Qdrant (COSINE)
```

- 컬렉션 `law_articles`(조문 단위)와 `law_naive500`(500자, 비교용) 두 개
- **청크 내용이 같으면 다시 임베딩하지 않는다**: (모델 + 청크 텍스트) 지문을 저장해 두고 비교 → 크레딧 절약

In [11]:
from rag.retriever import get_retriever
from rag import vectorstore as vs

r = get_retriever()   # 조문 청크 컬렉션 확인 (내용이 같으면 임베딩 생략)
r.naive([0.0] * 1536) # 500자 컬렉션도 준비

client = vs.client()
for name in (vs.STRUCT, vs.NAIVE):
    print(name, "→", client.count(name).count, "개 벡터")

law_articles → 57 개 벡터
law_naive500 → 57 개 벡터


In [12]:
# 실습 질문 4개의 벡터 — 처음 한 번만 임베딩해서 캐시한다 (원 질문 + Rewritten Query + Sub-query를 요청 1번에)
from rag import pipeline as P

CACHE = Path("notebook/.cache/query_vecs.json")
CACHE.parent.mkdir(parents=True, exist_ok=True)
VEC = json.loads(CACHE.read_text()) if CACHE.exists() else {}

def rewrite_of(gid):
    # LIVE면 LLM으로 새로 Query Rewriting, 아니면 평가 때 저장한 Query Rewriting 사용
    return P.rewrite(GOLDEN[gid]["question"]) if LIVE else DETAILS[gid]["rewrite"]

RW = {gid: rewrite_of(gid) for gid in DEMO}
for gid in DEMO:
    if LIVE or gid not in VEC:
        VEC[gid] = vs.embed_queries(P.queries(GOLDEN[gid]["question"], RW[gid]))
CACHE.write_text(json.dumps(VEC))
print({gid: f"질의 {len(v)}개 × {len(v[0])}차원" for gid, v in VEC.items()})

{'g02': '질의 5개 × 1536차원', 'g04': '질의 5개 × 1536차원', 'g16': '질의 5개 × 1536차원', 'g12': '질의 5개 × 1536차원'}


In [13]:
# 이후 단계에서 계속 쓸 출력 도우미: 정답 조문은 [대괄호]로 표시
def show(title, ranked_articles, gold, k=5):
    top = ranked_articles[:k]
    hit = len(set(gold) & set(top))
    marks = " ".join(f"[{n}]" if n in gold else n for n in top)
    print(f"{title:28} {marks:40} 정답 {hit}/{len(gold)}")

# 5. Dense 검색 — 500자 분할(S0) vs 조문 단위(S1)

Dense 검색은 질문과 청크를 같은 벡터 공간에 놓고 **가까운 것**을 찾는 의미 기반 검색이다.

```text
질문 ──임베딩──▶ 벡터 ──Qdrant 코사인 유사도──▶ 상위 청크 ──▶ 조문 번호
```

S0과 S1은 **검색 방법은 같고 청크만 다르다.** Chunking의 효과만 따로 볼 수 있다.

In [14]:
for gid in DEMO:
    g, q_vec = GOLDEN[gid], VEC[gid][0]          # VEC[gid][0] = 원 질문 벡터
    print(f"\n{gid} {g['question']}")
    show("500자 분할 + Dense (S0)", r.naive(q_vec), g["gold_articles"])
    show("조문 단위 + Dense (S1)", r.to_articles(r.dense(q_vec)), g["gold_articles"])


g02 우리 가게 상담 챗봇이 AI라는 걸 손님한테 미리 말해줘야 해? 안 하면 어떻게 돼?
500자 분할 + Dense (S0)         8 2 35 36 [31]                           정답 1/2
조문 단위 + Dense (S1)           2 40 [31] 32 33                          정답 1/2

g04 은행 AI가 내 대출을 거절했는데, 왜 그런지 설명해 달라고 할 수 있어?
500자 분할 + Dense (S0)         8 35 36 21 22                            정답 0/3
조문 단위 + Dense (S1)           36 40 32 25 17의2                         정답 0/3

g16 고영향 인공지능에 해당하는 영역에는 어떤 것들이 있어?
500자 분할 + Dense (S0)         30 31 34 35 33                           정답 0/1
조문 단위 + Dense (S1)           33 35 34 31 30                           정답 0/1

g12 해외 AI 사업자가 국내대리인을 안 두면 어떻게 되고, 국내대리인은 구체적으로 어떤 일을 대신해?
500자 분할 + Dense (S0)         35 [36] 37 38 21                         정답 1/5
조문 단위 + Dense (S1)           [36] 21 16 [32] 23                       정답 2/5


### 무엇이 보이나

- 500자 청크는 경계에 걸친 조문 2~3개를 한꺼번에 "검색됨"으로 세기 때문에 **지표상으로는 S0이 유리**하다. 그런데 실제로 LLM이 받는 건 잘린 조각이다.
- 그래서 검색 지표만 보지 말고 **Context(9장)와 답변(10장)까지 같이 본다.**

# 6. BM25 + RRF — Hybrid 검색 (S2)

임베딩은 "제36조", "국내대리인" 같은 **번호·고유명사를 뭉갠다.** 단어가 그대로 겹치는지를 보는 BM25를 같이 쓴다.

```text
                 ┌─ Dense (의미)   → 순위 A
질문 ────────────┤                          ─▶ RRF(순위 합치기) ─▶ 최종 순위
                 └─ BM25 (단어)    → 순위 B
```

- 한국어는 조사가 붙어("국내대리인을", "국내대리인은") 단어가 잘 안 겹친다 → **글자 2개씩 자른 bigram**을 토큰에 추가 (형태소 분석기 없이)
- Dense 점수(0~1)와 BM25 점수(0~수십)는 척도가 달라 더할 수 없다 → **점수 대신 순위**를 합치는 RRF

$$RRF(d) = \sum_{\text{검색기}} \frac{1}{k + \text{순위}(d)},\quad k = 60$$

In [15]:
from rag.retriever import tokenize, rrf

print(tokenize("국내대리인을 안 두면?"))

# RRF 손으로 계산해 보기: 문서 7은 두 검색기 모두 2위, 문서 3은 한쪽 1위·한쪽 없음
dense_rank, bm25_rank = [3, 7, 9], [5, 7, 1]
print("RRF 결과:", rrf([dense_rank, bm25_rank]))  # 두 곳에서 고르게 높은 7이 1위

['국내', '내대', '대리', '리인', '인을', '을안', '안두', '두면', '면?', '국내대리인을', '안', '두면?']
RRF 결과: [7, 3, 5, 9, 1]


In [16]:
for gid in DEMO:
    g, q_vec, q = GOLDEN[gid], VEC[gid][0], GOLDEN[gid]["question"]
    print(f"\n{gid} {q}")
    show("Dense만 (S1)", r.to_articles(r.dense(q_vec)), g["gold_articles"])
    show("   BM25만", r.to_articles(r.sparse(q)), g["gold_articles"])
    show("Dense + BM25 → RRF (S2)", r.to_articles(r.hybrid(q, q_vec)), g["gold_articles"])


g02 우리 가게 상담 챗봇이 AI라는 걸 손님한테 미리 말해줘야 해? 안 하면 어떻게 돼?
Dense만 (S1)                  2 40 [31] 32 33                          정답 1/2
   BM25만                     9 22의2 2 1 3                             정답 0/2
Dense + BM25 → RRF (S2)      2 9 3 22의2 1                             정답 0/2

g04 은행 AI가 내 대출을 거절했는데, 왜 그런지 설명해 달라고 할 수 있어?
Dense만 (S1)                  36 40 32 25 17의2                         정답 0/3
   BM25만                     [34] [3] 10 14 22                        정답 2/3
Dense + BM25 → RRF (S2)      36 22 [34] 40 37                         정답 1/3

g16 고영향 인공지능에 해당하는 영역에는 어떤 것들이 있어?
Dense만 (S1)                  33 35 34 31 30                           정답 0/1
   BM25만                     33 31 36 35 34                           정답 0/1
Dense + BM25 → RRF (S2)      33 35 31 34 30                           정답 0/1

g12 해외 AI 사업자가 국내대리인을 안 두면 어떻게 되고, 국내대리인은 구체적으로 어떤 일을 대신해?
Dense만 (S1)                  [36] 21 16 [32] 23                       정답 2/5
   BM25만                     [36] 

### 무엇이 보이나

- BM25는 질문의 **단어**가 법 본문에 있을 때 강하다. 그런데 "우리 가게 챗봇", "대출을 거절" 같은 **일상어**는 법 본문 어휘("인공지능사업자", "고지", "설명")와 겹치지 않아 오히려 순위를 흐릴 수 있다.
- 질문에 `제N조`가 직접 나오면 그 조문을 맨 앞으로 올린다(정규식 부스트) — 예: "제36조가 뭐야?"

# 7. Query Rewriting + Multi-Query (S3)

문제는 **질문의 말투**다. 사용자는 법률 용어를 모른다.

```text
"우리 가게 상담 챗봇이 AI라는 걸 손님한테 미리 말해줘야 해? 안 하면 어떻게 돼?"
        │  LLM 1회
        ▼
Rewritten Query: "상담 챗봇이 인공지능에 해당하는 경우 이용자에게 사전 고지 의무 …"
Sub-query: ① 사전 고지 의무  ② 미고지 시 과태료  ③ 이용자 보호 …
        │  각각 Hybrid 검색
        ▼
      RRF로 합침
```

- **검색에는 Query Rewriting을, 답변에는 원 질문을** 쓴다 (Query Rewriting이 질문 의도를 바꾸는 위험 차단)
- 쟁점이 둘인 질문("안 하면 어떻게 돼?")은 Sub-query로 나눠 **양쪽 다** 찾는다

In [17]:
for gid in DEMO:
    print(f"\n{gid} {GOLDEN[gid]['question']}")
    print("  Query Rewriting :", RW[gid]["rewrite"])
    for s in RW[gid]["subqueries"]:
        print("  Sub-query:", s)


g02 우리 가게 상담 챗봇이 AI라는 걸 손님한테 미리 말해줘야 해? 안 하면 어떻게 돼?
  Query Rewriting : 상담 챗봇이 인공지능에 해당하는 경우 이용자에게 사전 고지 의무가 있는지 및 미고지 시 제재 또는 법적 효과
  Sub-query: 상담 챗봇이 인공지능 시스템에 해당할 때 이용자에 대한 사전 고지 의무
  Sub-query: 인공지능 이용 사실 미고지 시 과태료 등 제재 규정
  Sub-query: 인공지능 상담서비스 제공자의 이용자 보호 또는 설명의무 관련 규정

g04 은행 AI가 내 대출을 거절했는데, 왜 그런지 설명해 달라고 할 수 있어?
  Query Rewriting : 인공지능 시스템의 자동화된 의사결정에 따른 대출 거절에 대하여 이용자가 결정 이유의 설명 또는 정보 제공을 요구할 수 있는지
  Sub-query: 인공지능 시스템의 자동화된 의사결정에 따른 금융거래 거절 시 이용자의 설명 요구권
  Sub-query: 인공지능에 의한 자동화된 대출 심사·거절 결과에 대한 통지 및 설명 제공 의무
  Sub-query: 인공지능 시스템이 처리한 개인신용·대출 관련 자동화된 결정에 대한 이의제기 또는 재검토 요청 가능 여부

g16 고영향 인공지능에 해당하는 영역에는 어떤 것들이 있어?
  Query Rewriting : 고영향 인공지능의 해당 영역
  Sub-query: 고영향 인공지능의 해당 영역은 무엇인가
  Sub-query: 고영향 인공지능에 포함되는 분야 또는 적용 영역은 무엇인가
  Sub-query: 고영향 인공지능 해당 여부를 판단하는 영역 기준은 무엇인가

g12 해외 AI 사업자가 국내대리인을 안 두면 어떻게 되고, 국내대리인은 구체적으로 어떤 일을 대신해?
  Query Rewriting : 해외 인공지능 사업자의 국내대리인 지정 의무와 국내대리인 미지정 시 제재, 그리고 국내대리인의 업무 범위
  Sub-query: 해외 인공지능 사업자가 국내대리인을 두지 아니한 경우의 제재 또는 법적 효과
  S

In [18]:
S3 = {}
for gid in DEMO:
    g, q = GOLDEN[gid], GOLDEN[gid]["question"]
    S3[gid] = r.multi(q, P.queries(q, RW[gid]), VEC[gid])   # 질의마다 Hybrid → RRF
    print(f"\n{gid} {q}")
    show("Hybrid, 원 질문만 (S2)", r.to_articles(r.hybrid(q, VEC[gid][0])), g["gold_articles"])
    show("+ Query Rewriting·Multi-Query (S3)", r.to_articles(S3[gid]), g["gold_articles"])


g02 우리 가게 상담 챗봇이 AI라는 걸 손님한테 미리 말해줘야 해? 안 하면 어떻게 돼?
Hybrid, 원 질문만 (S2)           2 9 3 22의2 1                             정답 0/2
+ Query Rewriting·Multi-Query (S3) [31] 40 33 32 13                         정답 1/2

g04 은행 AI가 내 대출을 거절했는데, 왜 그런지 설명해 달라고 할 수 있어?
Hybrid, 원 질문만 (S2)           36 22 [34] 40 37                         정답 1/3
+ Query Rewriting·Multi-Query (S3) 31 40 36 [34] 33                         정답 1/3

g16 고영향 인공지능에 해당하는 영역에는 어떤 것들이 있어?
Hybrid, 원 질문만 (S2)           33 35 31 34 30                           정답 0/1
+ Query Rewriting·Multi-Query (S3) 33 35 34 31 30                           정답 0/1



g12 해외 AI 사업자가 국내대리인을 안 두면 어떻게 되고, 국내대리인은 구체적으로 어떤 일을 대신해?
Hybrid, 원 질문만 (S2)           [36] 21 22 [32] 2                        정답 2/5
+ Query Rewriting·Multi-Query (S3) [36] [32] 4 21 23                        정답 2/5


# 8. Reranking (S4)

후보 20개 안에 정답이 **있는데도** 5위 밖으로 밀리는 경우가 있다.
임베딩은 질문과 문서를 **따로** 벡터로 만들어 비교하므로 세밀한 관련도를 못 본다.

```text
[Bi-encoder: 임베딩]                 [Cross-encoder: Reranker]
질문 → 벡터 ┐                         (질문, 조문) 한 쌍을 같이 읽고
조문 → 벡터 ┴→ 코사인 유사도           → "이 조문이 답에 도움이 되나?" 점수
빠름, 대충                            느림, 정확 → 후보 20개에만 적용
```

`Qwen/Qwen3-Reranker-0.6B`를 로컬 GPU에서 돌린다 (크레딧 0). 원 질문 기준으로 20 → 5.

In [19]:
from rag.reranker import rerank

chunk_texts = [c["text"] for c in r.chunks]
S4, SCORES = {}, {}
for gid in DEMO:
    g, q = GOLDEN[gid], GOLDEN[gid]["question"]
    scored = rerank(q, S3[gid][:20], chunk_texts)             # [(청크 번호, 점수)] 점수 높은 순
    S4[gid] = r.boost(q, [i for i, _ in scored])
    SCORES[gid] = scored
    print(f"\n{gid} {q}")
    show("Reranking 전 (S3)", r.to_articles(S3[gid]), g["gold_articles"])
    show("Reranking 후 (S4)", r.to_articles(S4[gid]), g["gold_articles"])

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Default prompt name is set to 'query'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.



g02 우리 가게 상담 챗봇이 AI라는 걸 손님한테 미리 말해줘야 해? 안 하면 어떻게 돼?
Reranking 전 (S3)             [31] 40 33 32 13                         정답 1/2
Reranking 후 (S4)             [31] 34 2 3 13                           정답 1/2



g04 은행 AI가 내 대출을 거절했는데, 왜 그런지 설명해 달라고 할 수 있어?
Reranking 전 (S3)             31 40 36 [34] 33                         정답 1/3
Reranking 후 (S4)             [2] [3] 16 [34] 33                       정답 3/3



g16 고영향 인공지능에 해당하는 영역에는 어떤 것들이 있어?
Reranking 전 (S3)             33 35 34 31 30                           정답 0/1
Reranking 후 (S4)             [2] 33 3 35 8                            정답 1/1



g12 해외 AI 사업자가 국내대리인을 안 두면 어떻게 되고, 국내대리인은 구체적으로 어떤 일을 대신해?
Reranking 전 (S3)             [36] [32] 4 21 23                        정답 2/5
Reranking 후 (S4)             [36] 39 [34] 2 4                         정답 2/5


In [20]:
# Reranker 점수 들여다보기 — 정답 조문(★)이 위로 올라왔나
gid = "g16"
print(GOLDEN[gid]["question"], "\n")
for i, s in SCORES[gid][:8]:
    c = r.chunks[i]
    star = "★" if c["no"] in GOLDEN[gid]["gold_articles"] else " "
    print(f"{star} {s:6.2f} {'█' * max(int((s + 6) * 2), 0):30} {c['label']}")

고영향 인공지능에 해당하는 영역에는 어떤 것들이 있어? 

★   6.12 ████████████████████████       제2조4
    1.38 ██████████████                 제33조
★  -0.38 ███████████                    제2조2
   -0.44 ███████████                    제3조
   -0.69 ██████████                     제35조
★  -1.12 █████████                      제2조11
   -1.50 █████████                      제8조
   -1.56 ████████                       제34조


### 시행착오 — Reranker 점수로 "범위 밖 질문"을 막으려다 실패

이전 실험에서 "Reranker 최고 점수가 0.91보다 낮으면 답하지 않는다"는 차단을 넣었다.
→ 범위 **안**의 일상어·multi-hop 질문 18개 중 **6개를 잘못 거절**, 정확도 90% → 65%.
일상어 질문은 법 조문과 표현이 달라 점수가 낮게 나올 뿐, 범위 밖이 아니었다.

**결론**: 점수로 자르지 않는다. 거절은 답변 프롬프트의 규칙("Context에 없으면 확인할 수 없다고 답하라")에 맡기고, 점수는 화면의 '검색 과정 보기'에 보여 준다.

# 9. Context 만들기 — 정의·참조·역참조 (C0 → C1 → C2)

LLM에 실제로 넣는 글이다. 상위 5개 조문은 **원문 그대로** 넣는다 (요약하면 "노력하여야 한다"가 "해야 한다"로 바뀐다).

| 단계 | 추가로 붙이는 것 | 예 |
|---|---|---|
| 상위 5개만 (C0) | 상위 5개 조문 | |
| 정의·참조 (C1) | + 질문에 나온 **정의어의 정의** | "고영향 인공지능" → 제2조4 |
| | + 본문이 가리키는 조문(**정방향 참조**) | 제43조 "제36조제1항을 위반하여" → 제36조① |
| 역참조 (C2) | + 상위 조문을 가리키는 **제재 조문(역참조)** | 제36조 → 제43조①2 "3천만원 이하 과태료" |

### 왜 역참조가 필요한가

```text
제36조(국내대리인 지정) ── "어기면?"에 대한 말이 없음
        ▲
        │ 가리킴
제43조(과태료) ①2. "제36조제1항을 위반하여 국내대리인을 지정하지 아니한 자" → 3천만원 이하
```

정방향 참조는 "A가 가리키는 B"만 가져온다. 이 법은 **제재 조문이 의무 조문을 가리키는** 구조라서,
"국내대리인 안 두면?"에 제36조를 찾아도 제43조는 따라오지 않았다 → 이전 실험의 가장 큰 약점.

In [21]:
# 역색인: "누가 나를 가리키나" — 제재·조사 조문(제40·42·43조)에서 자동으로 만든다
REV = P._law()["rev"]
for no in ["31", "36", "34", "7"]:
    print(f"제{no}조 ◀── " + ", ".join(u["label"] for u in REV.get(no, [])))

제31조 ◀── 제40조①1, 제40조①2, 제43조①1
제36조 ◀── 제43조①2
제34조 ◀── 제40조①1, 제40조①2
제7조 ◀── 제42조


In [22]:
gid = "g12"
q, top = GOLDEN[gid]["question"], r.to_articles(S4[gid])[:5]
print(q)
print("정답 조문:", GOLDEN[gid]["gold_articles"], "\n")

for name, kw in [("정의·정방향 참조 (C1)", {"reverse": False}), ("+ 역참조 (C2)", {})]:
    ctx, meta = P.build_context(q, RW[gid], top, **kw)
    have = sorted(set(top + meta["attached_articles"]) & set(GOLDEN[gid]["gold_articles"]), key=lambda x: int(re.match(r"\d+", x).group()))
    print(f"[{name}] Context {len(ctx):,}자 · 정답 조문 {len(have)}/{len(GOLDEN[gid]['gold_articles'])} 포함 {have}")
    print("   정의:", meta["definitions"], "| 참조:", meta["references"], "| 역참조:", meta["reverse_references"])

print("\n--- 역참조로 붙은 부분 (조문 전체가 아니라 해당 호만) ---")
print(ctx[ctx.find("[역참조"):][:400])

해외 AI 사업자가 국내대리인을 안 두면 어떻게 되고, 국내대리인은 구체적으로 어떤 일을 대신해?
정답 조문: ['36', '43', '32', '33', '34'] 

[정의·정방향 참조 (C1)] Context 6,267자 · 정답 조문 4/5 포함 ['32', '33', '34', '36']
   정의: [] | 참조: ['제32조②', '제33조①', '제13조', '제15조②', '제15조③', '제18조'] | 역참조: []
[+ 역참조 (C2)] Context 6,780자 · 정답 조문 5/5 포함 ['32', '33', '34', '36', '43']
   정의: [] | 참조: ['제32조②', '제33조①', '제13조', '제15조②', '제15조③', '제18조'] | 역참조: ['제43조①2', '제40조①1', '제40조①2']

--- 역참조로 붙은 부분 (조문 전체가 아니라 해당 호만) ---
[역참조 · 제43조①2(과태료) → 제36조]
① 다음 각 호의 어느 하나에 해당하는 자에게는 3천만원 이하의 과태료를 부과한다.
2. 제36조제1항을 위반하여 국내대리인을 지정하지 아니한 자

[역참조 · 제40조①1(사실조사 등) → 제34조]
① 과학기술정보통신부장관은 다음 각 호의 어느 하나에 해당하는 경우에는 인공지능사업자에 대하여 관련 자료를 제출하게 하거나, 소속 공무원으로 하여금 필요한 조사를 하게 할 수 있다.
1. 제31조제2항ㆍ제3항, 제32조제1항ㆍ제2항 또는 제34조제1항에 위반되는 사항을 발견하거나 혐의가 있음을 알게 된 경우

[역참조 · 제40조①2(사실조사 등) → 제34조]
① 과학기술정보통신부장관은 다음 각 호의 어느 하나에 해당하는 경우에는 인공지능사업자에 대하여 관련 


# 10. Grounded Answer + 출처

답변 프롬프트의 규칙:

1. Context에 있는 내용만으로 답한다
2. 주장마다 `[제34조①]`, `[제43조①1]`처럼 **조·항·호**를 인용한다
3. Context에 없으면 "제공된 조문에서는 확인할 수 없습니다"라고만 답한다
4. "대통령령으로 정한다"(시행령 위임)는 표시한다
5. "노력하여야 한다"(노력의무)와 "하여야 한다"(의무)를 구분한다

답변의 인용을 정규식으로 읽어 **그 단위의 원문**을 `sources`에 붙인다 → 사람이 1분 안에 대조할 수 있다.

In [23]:
# 인용 해석 — 모델이 형식을 섞어 써도 같은 단위로 맞춘다
for s in P.parse_sources("과태료 대상이에요[제43조①2]. 대출 심사는 고영향이에요[제2조제4호사목]. 없는 조문[제99조]."):
    print(s["article"], "→", s["content"].replace("\n", " ")[:70])

제43조①2 → ① 다음 각 호의 어느 하나에 해당하는 자에게는 3천만원 이하의 과태료를 부과한다. 2. 제36조제1항을 위반하여 국내대리인을
제2조4 → 이 법에서 사용하는 용어의 뜻은 다음과 같다. 4. “고영향 인공지능”이란 사람의 생명, 신체의 안전 및 기본권에 중대한 영향


In [24]:
gid = "g02"
if LIVE:
    c0 = credits()["remaining_credits"]
    out = P.ask(GOLDEN[gid]["question"])
    answer, sources = out["answer"], [s["article"] for s in out["sources"]]
    print("이번 셀 사용 크레딧:", c0 - credits()["remaining_credits"])
elif "ans" in DETAILS.get(gid, {}):
    answer, sources = DETAILS[gid]["ans"]["text"], DETAILS[gid]["ans"]["sources"]
else:
    answer, sources = None, None
    print("저장된 답변이 없어요. LIVE = True로 바꾸거나 eval/evaluate.py --answers 를 실행하세요.")

if answer:
    print("질문:", GOLDEN[gid]["question"], "\n")
    print(answer, "\n")
    print("출처:", sources)
    print("정답 요지:", GOLDEN[gid]["reference_answer"])

저장된 답변이 없어요. LIVE = True로 바꾸거나 eval/evaluate.py --answers 를 실행하세요.


# 11. Golden Test Set 40문항으로 전체 비교

질문 4개로는 우연일 수 있다. Golden Test Set 40문항(`docs/eval_set/`)으로 단계마다 쟀다.

| 지표 | 뜻 |
|---|---|
| Hit@3 | 상위 3개 조문에 정답 조문이 하나라도 있나 |
| Recall@5 | 정답 조문 중 몇 %가 상위 5개에 들었나 |
| MRR | 첫 정답 조문이 몇 위인가 (1위=1, 2위=0.5 …) |
| Context Recall | LLM에 실제로 넣은 Context 전체에 정답 조문이 몇 % 들어갔나 (Recall@5의 범위를 Context 전체로 넓힌 것) |

질문 유형 중 **multi-hop**은 조문 여러 개를 이어 봐야 답이 나오는 질문이다 (예: "국내대리인을 안 두면?" → 의무 제36조 + 과태료 제43조).
답변 표의 **할루시네이션**은 원문에 없거나 원문과 다른 내용을 이 법의 내용처럼 단정한 답변의 비율이다.
답변 채점은 이전 실험(g01~g20) 결과다 — 크레딧을 아끼려고 이번엔 다시 채점하지 않았다.

결과는 `uv run python eval/evaluate.py`가 만든 `eval/results.json`에서 읽는다 (이 셀은 크레딧 0).

In [25]:
R = json.load(open("eval/results.json"))
bar = lambda v: "█" * round(v * 20) + "░" * (20 - round(v * 20))

print(f"범위 안 {R['golden']['in_scope']}문항 기준\n")
print(f"{'단계':4} {'설명':24} {'Recall@5':>30}  Hit@3   MRR")
for s, v in R["stages"].items():
    print(f"{v['label'].removeprefix('+ ') + f' ({s})':34} {bar(v['recall5'])} {v['recall5']:4.0%}  {v['hit3']:4.0%}  {v['mrr']:.2f}")

print()
for c, v in R["context"].items():
    print(f"{v['label'].removeprefix('+ ') + f' ({c})':34} {bar(v['recall'])} {v['recall']:4.0%}  (정답 조문 전부 포함 {v['all_gold']:.0%})")

범위 안 35문항 기준

단계   설명                                             Recall@5  Hit@3   MRR
500자 고정 분할 + Dense (S0)            ████████████░░░░░░░░  61%   66%  0.58
조문 단위 Chunking (S1)                ███████████████░░░░░  75%   83%  0.78
Hybrid(BM25·RRF) (S2)              ██████████████░░░░░░  71%   89%  0.80
Query Rewriting·Multi-Query (S3)   ████████████████░░░░  79%   94%  0.94
Reranking (S4)                     █████████████████░░░  84%  100%  1.00

상위 5개 조문만 (C0)                     █████████████████░░░  84%  (정답 조문 전부 포함 71%)
정의·정방향 참조 (C1)                     ██████████████████░░  90%  (정답 조문 전부 포함 80%)
역참조(제재·조사 조문) (C2)                 ████████████████████  98%  (정답 조문 전부 포함 94%)


In [26]:
# 질문 유형별로 어느 단계가 효과가 컸나 (Recall@5)
stages = list(R["stages"])
SHORT = {"S0": "500자", "S1": "Chunk", "S2": "Hybrid", "S3": "Rewrite", "S4": "Rerank", "C2": "역참조"}
print(f"{'유형':12}", *[f"{SHORT[s]:>8}" for s in stages], f"{SHORT['C2']:>8}")
for t in R["stages"]["S0"]["by_type"]:
    row = [R["stages"][s]["by_type"][t]["recall5"] for s in stages] + [R["context"]["C2"]["by_type"][t]]
    print(f"{t:12}", *[f"{v:8.0%}" for v in row])

유형               500자    Chunk   Hybrid  Rewrite   Rerank      역참조
multi-hop         32%      52%      52%      54%      47%      93%
일상어               75%      75%      65%      86%      96%     100%
정의/세부             71%      86%      86%      86%     100%     100%
조문번호/고유명사         64%      93%      93%      93%      93%     100%


In [27]:
A, B = R.get("answers"), R["baseline_20"]["answers"]
names = {"A": "LLM 단독", "B": "법 전문 통째로", "C": "Naive RAG", "D_nogate": "Advanced RAG(역참조 없음)"}  # D는 점수 차단 버전(폐기)
print(f"{'방식':22} {'정확도':>6} {'할루시네이션':>6} {'인용정확':>8} {'입력토큰':>8}")
for k, n in names.items():
    v = B[k]
    print(f"{n:22} {v['accuracy']:6.0%} {v['hallucination_rate']:6.0%} {v['citation_valid_rate']:8.0%} {v['avg_input_tokens']:8,}  (이전 실험 20문항)")
if A:
    print(f"{'이번 파이프라인':22} {A['accuracy']:6.0%} {A['hallucination_rate']:6.0%} {A['citation_valid_rate']:8.0%} {int(A['avg_input_tokens']):8,}  ({A['n']}문항)")

방식                        정확도 할루시네이션     인용정확     입력토큰
LLM 단독                    35%    90%      17%      127  (이전 실험 20문항)
법 전문 통째로                  98%     0%     100%   24,355  (이전 실험 20문항)
Naive RAG                 65%    35%      60%    1,475  (이전 실험 20문항)
Advanced RAG(역참조 없음)      90%     0%     100%    4,483  (이전 실험 20문항)


# 12. 핵심 정리

```text
원문 HWPX       → XML이라 구조가 이미 있다. 파싱 결과는 Open API와 글자 단위로 대조
조문 단위 Chunking   → 정의·목록이 잘리지 않는다
BM25 + RRF      → 번호·고유명사를 놓치지 않는다 (점수 대신 순위로 합친다)
Query Rewriting·Multi-Query  → 일상어 질문을 법률 용어로 바꾸고, 쟁점을 나눠 다 찾는다
Reranking          → 후보 안의 정답을 상위로 끌어올린다 (점수로 거절하지는 않는다)
정의·참조·역참조 → 검색이 못 가져온 조문을 법의 구조로 채운다. 특히 "어기면?"의 제재 조문
Grounded Answer       → 조·항·호 인용 + 원문 출처, 없으면 없다고 말한다
Golden Test Set 평가      → 무엇을 바꿨을 때 실제로 나아졌는지 숫자로 확인한다
```

더 해 볼 것:
- `LIVE = True`로 바꾸고 `DEMO`에 직접 만든 질문 id를 넣어 보기 (Golden Test Set의 g21~g40도 있다)
- `rag/pipeline.py`의 `ENFORCEMENT`(역참조 대상 조문)나 `TOP_K`를 바꾸고 `uv run python eval/evaluate.py`로 다시 재기
- 전체 리포트: `uv run python eval/report.py` → `docs/report.html`, 서비스의 `/report`